In [1]:
from preprocessing.utils.defaults import AWS_REGION
import sagemaker
import boto3

sess = sagemaker.Session()

sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = sagemaker.get_execution_role()
except ValueError:
    iam = boto3.client('iam')
    role = iam.get_role(RoleName='sagemaker_execution_role')['Role']['Arn']

sess = sagemaker.Session(boto_session=boto3.Session(region_name=AWS_REGION), default_bucket=sagemaker_session_bucket)

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\karol\AppData\Local\sagemaker\sagemaker\config.yaml


Couldn't call 'get_role' to get Role ARN from role name superUser to get Role path.


sagemaker role arn: arn:aws:iam::767427092061:role/sagemaker_execution_role
sagemaker bucket: sagemaker-eu-central-1-767427092061
sagemaker session region: eu-west-1


In [2]:
from datasets import load_dataset


s3_dir =  's3://datalake-bucket-123/stages/$4b7047db-ab3c-4fca-811c-78069268dcae/ExplodeQuestionToQuestionChunkPair/results.parquet.gzip'

ds = load_dataset("parquet", data_files={"data": f"{s3_dir}/**/*.parquet"}, split="data")

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

In [3]:
from datasets import Dataset, DatasetDict


def build_datasets(type_of_chunking: str, test_size=0.15):
    if type_of_chunking == "article_based_chunking":
        # ------------------------------------------------------------------
        # 1️⃣  Build a unique-query list and split that list first
        # ------------------------------------------------------------------
        # create a mini-dataset with one column (id_query) and one row per query
        query_id_ds = Dataset.from_list(
            [{"id_query": qid} for qid in set(ds["id_query"])]
        )
        q_split = query_id_ds.train_test_split(test_size=test_size)  # keep it for testing
        train_qids = set(q_split["train"]["id_query"])
        test_qids = set(q_split["test"]["id_query"])

        train_ds = ds.filter(lambda ex: ex["id_query"] in train_qids and ex["type"] == "Article_Span")
        test_ds = ds.filter(lambda ex: ex["id_query"] in test_qids and ex["type"] != "Article")
        return train_ds, test_ds
    elif type_of_chunking == "legal_unit_chunking":
        # ------------------------------------------------------------------
        # 1️⃣  Keep only rows that are exact legal units which are need to answer question
        # ------------------------------------------------------------------
        full_ds_with_legal_unit_chunking = ds.filter(lambda ex: ex["contains_citation"] == 0)

        # ------------------------------------------------------------------
        # 1️⃣  Build a unique-query list and split that list first
        # ------------------------------------------------------------------
        # create a mini-dataset with one column (id_query) and one row per query
        query_id_ds = Dataset.from_list(
            [{"id_query": qid} for qid in set(full_ds_with_legal_unit_chunking["id_query"])]
        )
        q_split = query_id_ds.train_test_split(test_size=test_size)  # keep it for testing
        train_qids = set(q_split["train"]["id_query"])
        test_qids = set(q_split["test"]["id_query"])

        train_ds = full_ds_with_legal_unit_chunking.filter(lambda ex: ex["id_query"] in train_qids and ex["type"] != "Article_Span")
        test_ds = full_ds_with_legal_unit_chunking.filter(lambda ex: ex["id_query"] in test_qids and ex["type"] != "Article_Span")

        return train_ds, test_ds

Filter:   0%|          | 0/148332 [00:00<?, ? examples/s]

Filter:   0%|          | 0/148332 [00:00<?, ? examples/s]

In [ ]:

path_s = 'https://s3.console.aws.amazon.com/s3/buckets/sagemaker-eu-central-1-767427092061/?region=prefix=datasets/embedding_article_based_chunking/'

In [15]:

import boto3
s3 = boto3.client("s3")

def check_exists(bucket, key):
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except s3.exceptions.ClientError as e:
        if e.response['Error']['Code'] == "404":
            return False
        else:
            raise

In [5]:
input_path = f's3://sagemaker-eu-central-1-767427092061/datasets/embedding_article_based_chunking'

In [6]:
train_dataset_article_span_s3_path = f"{input_path}/train/dataset.json"
test_dataset_article_span_s3_path = f"{input_path}/test/dataset.json"

In [7]:
print(train_dataset_article_span_s3_path)

s3://sagemaker-eu-central-1-767427092061/datasets/embedding_article_based_chunking/train/dataset.json


In [11]:
bucket = "sagemaker-eu-central-1-767427092061"
train_key = "datasets/embedding_article_based_chunking/train/dataset.json"

In [16]:
print("Train exists:", check_exists(bucket, train_key))

Train exists: True


In [17]:
import time
from sagemaker.huggingface import HuggingFace

In [24]:


training_arguments = {
    "model_id": "sdadas/mmlw-retrieval-roberta-large-v2",  # model id from the hub
    "chunking_type":"article_based_chunking",
    "train_dataset_path": "/opt/ml/input/data/train/",  # path inside the container where the training data is stored
    "test_dataset_path": "/opt/ml/input/data/test/",  # path inside the container where the test data is stored
    "num_train_epochs": 5,  # number of training epochs
    "learning_rate": 2e-5,  # learning rate
    'per_device_train_batch_size': 6,  # batch size per device during training
    'per_device_eval_batch_size': 4,
    'gradient_accumulation_steps': 8,
    'eval_only': True
}

job_name = f'roberta-large-{time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())}-article-span-retrival'

# create the Estimator
huggingface_estimator = HuggingFace(
    entry_point='run_mnr.py',  # train script
    source_dir='scripts_pirb',  # directory which includes all the files needed for training
    instance_type='ml.g5.xlarge',  # instances type used for the training job
    instance_count=1,  # the number of instances used for training
    max_run=2 * 24 * 60 * 60,  # maximum runtime in seconds (days * hours * minutes * seconds)
    base_job_name=job_name,  # the name of the training job
    role=role,  # Iam role used in training job to access AWS ressources, e.g. S3
    transformers_version='4.36.0',  # the transformers version used in the training job
    pytorch_version='2.1.0',  # the pytorch_version version used in the training job
    py_version='py310',  # the python version used in the training job
    hyperparameters=training_arguments,
    disable_output_compression=True,  # not compress output to save training time and cost
    environment={
        "HUGGINGFACE_HUB_CACHE": "/tmp/.cache",  # set env variable to cache models in /tmp
    },
)

In [25]:
data = {
    'train': train_dataset_article_span_s3_path,
    'test': test_dataset_article_span_s3_path,
}

# starting the train job with our uploaded datasets as input
huggingface_estimator.fit(data, wait=True)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: roberta-large-2025-08-17-13-14-27-artic-2025-08-17-11-14-29-562


2025-08-17 11:14:31 Starting - Starting the training job
2025-08-17 11:14:31 Pending - Training job waiting for capacity...
2025-08-17 11:14:55 Pending - Preparing the instances for training...
2025-08-17 11:15:21 Downloading - Downloading input data...
2025-08-17 11:15:46 Downloading - Downloading the training image........................
2025-08-17 11:19:58 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
/opt/conda/lib/python3.10/site-packages/paramiko/pkey.py:100: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
/opt/conda/lib/python3.10/site-packages/paramiko/transport.py:259: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will b

In [26]:
input_path_exact_legal_unit = f's3://sagemaker-eu-central-1-767427092061/datasets/embedding_legal_unit_chunking'

In [27]:
train_ds_exact_legal_unit_s3_path = f"{input_path_exact_legal_unit}/train/dataset.json"
test_ds_exact_legal_unit_s3_path = f"{input_path_exact_legal_unit}/test/dataset.json"
train_ds_exact_legal_unit_s3_path

's3://sagemaker-eu-central-1-767427092061/datasets/embedding_legal_unit_chunking/train/dataset.json'

In [28]:
training_arguments_for_exact_legal_unit = {
    "model_id": "sdadas/mmlw-retrieval-roberta-large-v2",  # model id from the hub
    "chunking_type": "legal_unit_chunking",
    "train_dataset_path": "/opt/ml/input/data/train/",  # path inside the container where the training data is stored
    "test_dataset_path": "/opt/ml/input/data/test/",  # path inside the container where the test data is stored
    "num_train_epochs": 5,  # number of training epochs
    "learning_rate": 2e-5,  # learning rate
    'per_device_train_batch_size': 6,  # batch size per device during training
    'per_device_eval_batch_size': 4,
    'gradient_accumulation_steps': 8,
    'eval_only': True
}

job_name = f'roberta-{time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())}-exact-legal-unit'

# create the Estimator
huggingface_estimator_for_exact_legal_unit = HuggingFace(
    entry_point='run_mnr.py',  # train script
    source_dir='scripts_pirb',  # directory which includes all the files needed for training
    instance_type='ml.g5.xlarge',  # instances type used for the training job
    instance_count=1,  # the number of instances used for training
    max_run=2 * 24 * 60 * 60,  # maximum runtime in seconds (days * hours * minutes * seconds)
    base_job_name=job_name,  # the name of the training job
    role=role,  # Iam role used in training job to access AWS ressources, e.g. S3
    transformers_version='4.36.0',  # the transformers version used in the training job
    pytorch_version='2.1.0',  # the pytorch_version version used in the training job
    py_version='py310',  # the python version used in the training job
    hyperparameters=training_arguments_for_exact_legal_unit,
    disable_output_compression=True,  # not compress output to save training time and cost
    environment={
        "HUGGINGFACE_HUB_CACHE": "/tmp/.cache",  # set env variable to cache models in /tmp
    },
)

In [29]:
data_for_exact = {
    'train': train_ds_exact_legal_unit_s3_path,
    'test': test_ds_exact_legal_unit_s3_path,
}

# starting the train job with our uploaded datasets as input
huggingface_estimator_for_exact_legal_unit.fit(data_for_exact, wait=True)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: roberta-2025-08-17-16-27-44-exact-legal-2025-08-17-14-27-49-046


2025-08-17 14:27:50 Starting - Starting the training job
2025-08-17 14:27:50 Pending - Training job waiting for capacity...
2025-08-17 14:28:10 Pending - Preparing the instances for training...
2025-08-17 14:28:42 Downloading - Downloading input data...
2025-08-17 14:29:07 Downloading - Downloading the training image........................
2025-08-17 14:33:25 Training - Training image download completed. Training in progress...bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
/opt/conda/lib/python3.10/site-packages/paramiko/pkey.py:100: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
/opt/conda/lib/python3.10/site-packages/paramiko/transport.py:259: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will 